# NACC data importation, analysis and preparation

## Libraries and Data Files
- Import necessary libraries, `pandas` and `numpy`.
- Load several CSV files containing data and metadata for analysis.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
all_nacc = pd.read_csv('/SeaExpCIFS/NACC_0620/investigator_ftldlbd_nacc65.csv')
nacc_variable = pd.read_csv('../data/nacc_variable.csv')
nacc_allowable_code = pd.read_csv('../data/nacc_allowable_code.csv')

## Data Cleaning

- Fill missing values in variable_id column using forward filling method.

In [3]:
nacc_allowable_code['variable_id_conv'] = nacc_allowable_code['variable_id'].fillna(method='ffill')
nacc_allowable_code.rename(columns={"descriptor": "individual_descriptor"}, inplace=True)

## Data Merging

- Merge `nacc_variable` with `nacc_allowable_code` on variable IDs to consolidate variable descriptions and their corresponding codes.
- Filter merged data for specific UDS forms (D1 and D2) to get diagnosis features.

In [4]:
merged_df = pd.merge(nacc_variable, nacc_allowable_code, left_on='id', right_on='variable_id_conv', how='outer')
merged_df_diag = merged_df[(merged_df['form_id'] == 'D1') | (merged_df['form_id'] == 'D2')]

## Data Transformation

- Generate dictionaries for 
    - variable name and text diagnosis
    - descriptor codes and text descriptions for each code.

In [8]:
diag_dict = dict(zip(merged_df_diag['id'], merged_df_diag['descriptor']))
code_dict = {
    id_value: {row['code_1']: row['individual_descriptor'] for index, row in group.iterrows()}
    for id_value, group in merged_df_diag.groupby('id')
}

## Diagnosis Dictionary Creation

- Filter the diagnostic columns for NACC data.
- Iterate through filtered data to map NACC IDs to corresponding diagnostic texts.

In [10]:
filtered_nacc = all_nacc[list(diag_dict.keys()) + ["NACCID"]].replace({-4: np.NaN, "-4": np.NaN})
nacc_diag_dict = {}
for i, row, in filtered_nacc.iterrows():
    st = ""
    for k, v in dict(row).items():
        if k == "NACCID":
            continue
        try:
            if not pd.isna(v):
                st += f"{diag_dict[k].replace(';', ',')}: {code_dict[k][str(int(v))].replace(';', ',')}; "
        except:
            st += f"{diag_dict[k].replace(';', ',')}: {v}; "
            
    if row['NACCID'] in nacc_diag_dict:
        nacc_diag_dict[row['NACCID']].append(st)
    else:
        nacc_diag_dict[row['NACCID']] = [st]

## Data Export

- Save the diagnosis dictionary to a JSON file for further use.

In [12]:
import json

with open('../data/nacc_diagnosis_text.json', 'w', encoding='utf-8') as fp:
    json.dump(nacc_diag_dict, fp, sort_keys=True, indent=4, ensure_ascii=False)

# LLAMA 3

## Data Loading
- Load diagnostic texts from a JSON file into a Python dictionary.
- Extract diagnostic text entries and compile them into a single list for processing.

In [6]:
import json

with open('../data/nacc_diagnosis_text.json', 'r') as fp:
    data = json.load(fp)
    
# texts = list(data.values())
texts = []
for k,v in data.items():
    texts += v

## Language Model Setup

### Login to huggingface

In [4]:
from huggingface_hub import login
login()

### Model and Tokenizer Initialization

- Import necessary modules from the transformers library.
- Initialize the tokenizer and model for text generation using the Meta-Llama pre-trained model.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
# model_id = "meta-llama/Meta-Llama-3-70B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Text Generation Setup

- Define a list of messages simulating a conversation scenario where the model acts as a behavioral neurologist.
- Prepare the input tensor using the tokenizer, applying a chat template to mimic a dialogue context.
- Configure parameters for the language model's generation process, such as the number of tokens, sampling temperature, and top-p (nucleus) sampling.
- Decode the generated response and print it.

In [7]:
messages = [
    {"role": "system", "content": "For the given diagnostic information, your task is to act as a behavioral neurologist and summarize the diagnosis in 4 to 5 sentences using the 'present' tense. Focus on delivering a clear diagnostic summary based on the provided information, without mentioning the tests the subject underwent or the term 'UDS.' Ensure the diagnosis identifies the cognitive status, mentions any identified disorders, and specifies the primary etiological diagnosis."},
    {"role": "user", "content": data['NACC000271'][1]},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    input_ids,
    max_new_tokens=512,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
response = outputs[0][input_ids.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


The subject is currently experiencing mild cognitive impairment (MCI) with cognitive deficits affecting language, attention, and multiple domains. The primary etiological diagnosis is Alzheimer's disease (AD), which is considered the primary cause of their cognitive impairment. The subject's cognitive status is not normal, and they have progressed from MCI to dementia.


#### Example responses to patient ID: NACC000271

0. The individual is currently experiencing mild cognitive impairment (MCI), specifically amnestic MCI with impairment in multiple domains, including language. The primary etiological diagnosis is Alzheimer's disease (AD), which is considered the primary cause of their cognitive impairment. There is no evidence of other underlying conditions, such as primary progressive aphasia, frontotemporal dementia, Lewy body disease, Parkinson's disease, or Huntington's disease, that could be contributing to their cognitive impairment.

1. The subject is currently experiencing mild cognitive impairment (MCI), specifically Amnestic MCI with impairment in multiple domains, including language and attention. The primary cognitive disorder is presumed to be Alzheimer's disease (AD), which is considered the primary cause of the observed cognitive impairment. The subject has not progressed to dementia, and their cognitive status remains MCI.

2. The patient is diagnosed with Alzheimer's disease, which is the primary cause of their cognitive impairment. They have met criteria for dementia and have progressed from a state of normal cognition to a state of dementia. The patient's cognitive impairment is not due to any other underlying condition, including frontotemporal dementia, Lewy body dementia, vascular dementia, or other neurodegenerative disorders.

3. The subject is cognitively impaired and meets the criteria for dementia. The primary cause of cognitive impairment is Alzheimer's disease, which is a presumptive etiologic diagnosis. The subject's cognitive impairment is not due to any other underlying condition, including primary progressive aphasia, behavioral variant frontotemporal dementia, Lewy body dementia, vascular dementia, or other neurological or psychiatric disorders.

4. This individual is cognitively impaired and meets the criteria for dementia. The primary contributing cause of their cognitive impairment is Alzheimer's disease, which is presumed to be the underlying etiology of their cognitive decline. They do not have any other underlying medical or psychiatric conditions that could be contributing to their cognitive impairment.

5. The patient is diagnosed with probable Alzheimer's disease, with Alzheimer's disease being the primary cause of their cognitive impairment. They meet criteria for dementia and have a presumptive etiologic diagnosis of probable Alzheimer's disease. The patient's cognitive status has progressed from normal cognition to dementia during follow-up visits.